# 6.5 EU Angle: Data Residency, GDPR Logs, AI Act Basics

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

**Prerequisites:** 5.1 (persistence and checkpointing), 6.1 (LangSmith tracing), 6.3 (guardrails and PII redaction). No API key needed; the demos use a local SQLite checkpointer.

**A note before we start.** This lesson is engineering context, not legal advice. The rules summarised here are real and the dates are current, but applying them to a real product is a conversation with a lawyer, not a lesson. Treat everything here as "what to ask about", not "what to do".

## Learning objectives

- Choose a data residency posture for an agent service: what EU regions buy you, what they do not, and when self-hosting is the honest answer.
- Apply GDPR to agent logs: identify personal data inside traces and checkpoints, set retention windows, and implement the right to erasure.
- Describe the EU AI Act as it touches a deployer: provider versus deployer, the risk tiers, the transparency duties that are already in force, and the fines.
- Assemble a pre-launch compliance checklist for the capstone that ties the course's engineering (redaction, evals, budgets) to the legal duties.

## Why this lesson exists

This course's audience is European. Your users will be in Berlin, Paris, Warsaw. Their employers will ask where the data goes before they ask what the model is. And since August 2026, a wrong answer to "does your chatbot say it is a chatbot" carries a fine. Deployment without this lesson is deployment with a blind spot.

## Data residency: where the bytes sleep

Data residency is the promise that a customer's data stays in a chosen geography, usually the EU/EEA. It matters because customers in regulated sectors (health, finance, public sector) often cannot legally use a service whose data crosses into jurisdictions with weaker protections, and because procurement questionnaires ask the question explicitly.

You have three postures, in increasing order of control and cost.

**1. An EU region of a major cloud.** Every large provider has one: AWS `eu-central-1` (Frankfurt) and `eu-west-3` (Paris), Azure West Europe (Netherlands) and North Europe (Ireland), Google Cloud `europe-west3` (Frankfurt). Deploy the API container and the Postgres database in the same EU region. For most products this is the right answer: the data at rest and in transit stays in the EU, the provider's data processing addendum covers the subprocessing, and the operational cost is the same as any other region.

**2. EU region plus contractual controls.** Residency is necessary but not sufficient. Your model vendor is a subprocessor: when your agent calls an API, the prompt (with the user's data in it) travels to the vendor. Check where the vendor processes it. Most large vendors offer EU data processing or zero-retention options on their business tiers; the consumer tier often does not. Read the vendor's data processing terms before you promise residency to your own customers, because your promise covers the whole chain, not just your container.

**3. Self-hosting.** Run the models yourself (Ollama path from lesson 0.2, or a GPU host in an EU datacenter) so no third party ever sees the data. This is the honest answer for the strictest customers, and it is also the most expensive and operationally heavy: you now own GPU capacity, model updates, and uptime. Choose it when the contract requires it, not as a default.

What residency does *not* give you: it does not anonymise anything, it does not satisfy GDPR on its own, and it does not exempt you from the AI Act. It answers one question, "where", and the next two sections answer "how long" and "with what duties".

## GDPR for agent logs

Everything an agent touches is a GDPR event if it contains personal data. And agent logs are full of it: the user's name in the prompt, their email in a tool result, their order history in the retrieved context, their question quoted in the trace. Lesson 6.3 taught redaction before data reaches the model; this lesson is about the data you keep afterwards.

Three principles do most of the work.

**Lawfulness and minimisation.** Collect a legal basis (usually the contract with the user, or consent) and keep only what the product needs. A trace that stores the full prompt, the full tool outputs, and the full response is convenient for debugging and maximal for exposure. Decide per field: do you need the prompt text, or is the redacted form enough? The redacted form is almost always enough.

**Storage limitation.** Personal data lives only as long as it has a purpose. A support conversation needed for the current session is not needed two years later. Set a retention window and enforce it. Checkpoints are the concrete mechanism: a checkpoint row is a snapshot of a conversation, and it is personal data if the conversation contains any.

In [ ]:
# pip install langgraph-checkpoint-sqlite==3.1.2  (no API key needed)
from langgraph.checkpoint.sqlite import SqliteSaver

saver = SqliteSaver.from_conn_string("checkpoints.db")
saver.setup()

# Right to erasure, concretely: delete every checkpoint of one thread.
# Wire this to your "delete my data" endpoint.
saver.delete_thread("user-42:thread-7")
print("thread deleted")

`delete_thread` removes the conversation history for one thread. Your deletion endpoint should also cover the other places copies live: the vector store documents derived from the user's data, the LangSmith traces if you kept them past the debugging window, and any exported logs. Deletion that misses one store is not deletion.

Retention enforcement is the scheduled version of the same idea: a nightly job that deletes threads older than your policy window (say, 90 days for support chats). Postgres users get a `prune()` helper on the saver for exactly this; SQLite users write the equivalent query. The social-media war stories in this space all have the same shape: checkpoints were kept forever "for debugging", nobody owned the retention policy, and the table grew into millions of rows of personal data with no legal basis to keep them.

**Processor contracts.** Your model vendor, your cloud host, your tracing service: each is a processor of your users' data, and GDPR requires a data processing agreement with each. The business tier of the service usually provides one; the free tier often does not. This is a procurement checklist item, not a code item, which is why it lives in this lesson and not in 6.3.

## The EU AI Act, for the person deploying the agent

The AI Act entered into force in August 2024 and rolls out in phases. The phase that matters most to you is already live. Here is the shape of it.

**Provider versus deployer.** The *provider* builds the AI system; the *deployer* uses it in their operations. When you ship an agent on top of someone else's model, you are the deployer (sometimes also the provider of the system as a whole, when you brand and market it yourself; your lawyer draws that line). The duties below are the deployer's.

**Risk tiers.** The Act sorts systems into prohibited, high-risk, transparency-duty, and minimal-risk. Prohibited practices (social scoring, manipulative systems, and since December 2026, AI-generated non-consensual intimate imagery including "nudify" tools) have been banned since February 2025. High-risk systems (hiring, credit, education, law enforcement, and similar Annex III uses) face the heaviest duties, but their full obligations were deferred by the Digital Omnibus to December 2027. Most agents students build land in the third tier: not high-risk, but subject to transparency duties.

**The duties already in force for deployers.** Since 2 August 2026, Article 50 requires that people are told when they are interacting with an AI system. Your chat widget needs a visible disclosure ("You are chatting with an AI assistant"), unless it is obvious from context. AI-generated or manipulated content that looks like real people, places, or events must be labelled as AI-generated. AI-generated text on matters of public interest (health, safety, consumer advice) must be labelled unless a human has editorially reviewed it and takes visible responsibility. The provider-side watermarking duty follows on 2 December 2026; the deployer labelling duty is live now.

**The timeline as of October 2026.** Prohibited practices: in force since February 2025. General-purpose AI model duties (transparency, copyright, safety): applying since August 2025, with Commission enforcement and fines active since August 2026. Deployer transparency (Article 50): in force since 2 August 2026. New prohibitions on intimate-imagery abuse: December 2026. High-risk stand-alone systems (Annex III): December 2027. High-risk systems embedded in regulated products (Annex I): August 2028. Member-state regulatory sandboxes: August 2027. The Digital Omnibus (Regulation 2026/1744, in force since 27 July 2026) is what moved the high-risk dates; it did not move the transparency dates.

**Fines.** Up to 35 million EUR or 7% of global turnover for prohibited practices; up to 15 million EUR or 3% for most other breaches, including the transparency duties. Authorities must apply proportionality for SMEs, which softens but does not remove the exposure.

**What the deployer owes in practice.** Use the system according to the provider's instructions. Keep human oversight where the risk warrants it (lesson 5.2's approval gates are the engineering form of this). Monitor the system in operation and log incidents. Train the staff who use it; AI literacy has been a duty since February 2025. And understand that buying from a compliant vendor does not transfer your duties: the deployer obligations are yours even when the model is someone else's.

## The pre-launch checklist

For the capstone, this lesson compresses to a page you fill in before anyone outside the team touches the product:

1. **Residency decision:** EU region named, vendor data-processing terms checked, subprocessor list written down.
2. **Disclosure:** the chat UI says it is AI, in the user's language, before the first message.
3. **Redaction:** PII redaction from lesson 6.3 applied to prompts, tool outputs, and traces.
4. **Retention:** a written window (e.g. 90 days) and a job that enforces it; a working deletion path per thread.
5. **Oversight:** approval gates on consequential actions (lesson 5.2); an incident log someone actually reads.
6. **Evidence:** the eval suite from lesson 6.2, versioned with model and prompt metadata; an auditor who asks "is it accurate and robust" gets the experiment history, not a shrug.
7. **A lawyer:** has reviewed this page for the real product. This lesson has not.

## What can go wrong

**The disclosure nobody added.** The chatbot ships, works beautifully, and never says it is a chatbot. This is the most common AI Act breach for small teams: not a dramatic prohibited practice, just a missing sentence in the UI, live since August 2026 and fineable. Add it in the frontend, not in the model prompt.

**Logs that are a liability.** Full prompts and tool outputs in plaintext logs, kept forever, in a bucket nobody owns. One breach or one subject-access request turns this into the expensive part of the incident. Redact at write time; the data you never stored is the data you never have to defend.

**The vendor term nobody read.** The residency promise says "EU only", but the model vendor's consumer tier processes prompts in the US and retains them for training. Your promise covered the whole chain, so the breach is yours. Read the data processing terms on the tier you actually pay for.

**Retention theatre.** A policy document says 90 days; no job enforces it; the checkpoint table has rows from the beta. Auditors and regulators check enforcement, not documents. The nightly prune job is the policy.

**Confusing the high-risk delay with a general delay.** The Digital Omnibus moved high-risk dates to 2027/2028. It did not move transparency, literacy, or prohibited-practice duties. "The AI Act was delayed" is the sentence that gets teams fined; the transparency duties were not delayed.

## Check your understanding

**1. Your agent runs in AWS eu-central-1 and the database is in the same region. A customer asks whether their data stays in the EU. Why is "yes" the wrong complete answer?**

Because the model vendor is part of the chain: every prompt your agent sends travels to the vendor's API, and if the vendor processes or retains it outside the EU (common on consumer tiers), the data leaves the EU regardless of where your container runs. The complete answer requires checking the vendor's data processing terms on the tier you pay for, and naming the vendor as a subprocessor.

**2. A user invokes their right to erasure. You call `delete_thread` on the checkpointer. Name two other stores you must also clear, and explain why checkpoint deletion alone is insufficient.**

The vector store (documents or embeddings derived from the user's data) and the tracing/logging backend (LangSmith traces, exported logs) both hold copies of the user's personal data. Erasure means every copy, and the checkpoint is only one of the places the conversation was persisted. A deletion endpoint should enumerate all stores up front.

**3. Your capstone is a support chatbot for a Berlin retailer. It is not a high-risk system. Which AI Act duties apply to you as the deployer right now, and which dates matter?**

The Article 50 transparency duties, in force since 2 August 2026: disclose that users are interacting with AI, and label AI-generated content appropriately. The high-risk duties do not apply (and were deferred to December 2027 anyway). AI literacy for staff has applied since February 2025. Fines for transparency breaches run up to 15 million EUR or 3% of turnover.

**4. Why does buying a model from a vendor with full AI Act documentation not remove your deployer duties?**

Because the Act assigns duties to roles, and using the system in your operations makes you the deployer with your own obligations: use per the provider's instructions, maintain human oversight, monitor in operation, log incidents, train staff. The vendor's documentation helps you comply; it does not comply for you.

**5. Your team writes a retention policy of 90 days for conversation checkpoints but never builds the enforcement job. Eighteen months later, what is the concrete exposure?**

You hold personal data (conversation content) with no legal basis for keeping it past the stated purpose, which breaches the storage-limitation principle, and the checkpoint table has grown into a large, unowned liability. The exposure is both regulatory (the policy is theatre an auditor will see through) and practical (a breach or access request now covers eighteen months of data instead of ninety days).

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.